In [0]:
# 1. Define your credentials
storage_account_name = "shiwamdataproject01"
storage_account_key = "bNNQtSWcqFO6swEaxGWHRRJ2kZfYmo4bkgwcijVnoWmNeng+NSiFtZ2GPqwnMyX4LzIskpmgwKo2+ASt7y8uQg=="
container_name = "medallion" # The container you created in ADLS

# 2. Configure Spark to use the Access Key
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

# 3. Define the Base Path for your project
# Use 'abfss' (Azure Blob File System Driver - Secure)
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"

print(f"Connection set up for: {base_path}")

In [0]:
# %sql
# CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# # 1. The Raw Source Data (where the 750M 'broken' files are)
# source_path = f"{base_path}raw/transactions/"
# # 2. The Auto Loader Metadata (where the old schema is 'locked')
# schema_path = f"{base_path}checkpoints/bronze_fact/schema"
# # 3. The Stream Offsets (where Spark remembers what it has already read)
# offset_path = f"{base_path}checkpoints/bronze_fact/offsets"

# # --- Execute Deletions ---
# print(f"Deleting Raw Data: {source_path}")
# dbutils.fs.rm(source_path, recurse=True)
# print(f"Deleting Locked Schema: {schema_path}")
# dbutils.fs.rm(schema_path, recurse=True)
# print(f"Deleting Stream Offsets: {offset_path}")
# dbutils.fs.rm(offset_path, recurse=True)
# # 4. Drop the Table Metadata
# spark.sql("DROP TABLE IF EXISTS bronze.fact_transactions")
# print("\n--- Cleanup Complete! You can now run your updated Producer. ---")

In [0]:
# %sql
# -- 1. Remove the metadata from the Databricks Catalog
# DROP TABLE IF EXISTS bronze.fact_transactions;

# -- 2. (Optional but Recommended) If you want to be extra safe, 
# -- you can also run this to clear the cache
# CLEAR CACHE;

In [0]:
from pyspark.sql import functions as F

source_path = f"{base_path}raw/transactions/"
checkpoint_path = f"{base_path}checkpoints/bronze_fact/"

# Auto Loader Stream
bronze_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .load(source_path)
    .withColumn("ingestion_time", F.current_timestamp()))

# Write to Bronze Delta Table
query = (bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/offsets")
    .trigger(availableNow=True)
    .toTable("bronze.fact_transactions")) # This creates the table in your catalog

query.awaitTermination()
print("Ingestion Complete!")

In [0]:
%sql
select * from bronze.fact_transactions

In [0]:
%sql
select count(*) from bronze.fact_transactions

In [0]:
%sql
select store_id, count(*) as cnt from bronze.fact_transactions
group by all
order by cnt desc